# DeSPTD 收敛速度对比：调参后的正式实验

这个 notebook 是单文件版本，包含环境、算法、参数、绘图和结果保存。  
正式在线阶段的横轴是 **Step**，不计入预采样；所有方法每得到一个新样本都更新一次。

## 1. 导入基础库

Jupyter 和 Colab 通常已经安装 NumPy、Matplotlib。若这一格提示缺少库，再新建一格运行：

```python
%pip install numpy matplotlib
```

In [ ]:
from __future__ import annotations

import csv
import json
import math
import shutil
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ALGORITHMS = ("DeSPTD", "DeTDC", "DeTD(0)")
SCHEDULES = ("constant", "one_over_t")
METRICS = ("mean_parameter_error", "consensus_error", "max_agent_error")
COLORS = ("#087e8b", "#d45b1f", "#777777")
STYLES = ("-", "--", "-.")

print("基础库载入完成。NumPy 版本：", np.__version__)

## 2. 实验环境

这里定义论文实验使用的双智能体 Cliff Walking，以及一个用于诊断 off-policy 不稳定性的两状态环境。默认实验使用 Cliff Walking。

In [ ]:
class TwoAgentCliff:
    """Continuing two-agent 3x4 cliff-navigation environment."""

    height = 3
    width = 4
    actions = 4  # up, down, left, right
    agents = 2

    def __init__(self, target_epsilon: float):
        self.local_states = self.height * self.width
        self.states = self.local_states**self.agents
        self.start_local = self.pos_to_local(self.height - 1, 0)
        self.goal_local = self.pos_to_local(self.height - 1, self.width - 1)
        self.start_state = self.encode(self.start_local, self.start_local)

        self.next_state = np.zeros(
            (self.states, self.actions, self.actions), dtype=np.int16
        )
        self.reward = np.zeros(
            (self.states, self.actions, self.actions, self.agents), dtype=float
        )
        self._build_dynamics()

        self.behavior = np.full(
            (self.agents, self.states, self.actions), 1.0 / self.actions
        )
        self.target = self._build_target_policy(target_epsilon)

    def pos_to_local(self, row: int, col: int) -> int:
        return row * self.width + col

    def local_to_pos(self, local_state: int) -> tuple[int, int]:
        return divmod(local_state, self.width)

    def encode(self, local1: int, local2: int) -> int:
        return local1 * self.local_states + local2

    def decode(self, state: int) -> tuple[int, int]:
        return divmod(state, self.local_states)

    def _move(self, local_state: int, action: int) -> tuple[int, float]:
        if local_state == self.goal_local:
            return self.goal_local, -0.5

        row, col = self.local_to_pos(local_state)
        dr = (-1, 1, 0, 0)
        dc = (0, 0, -1, 1)
        row2 = min(self.height - 1, max(0, row + dr[action]))
        col2 = min(self.width - 1, max(0, col + dc[action]))

        if row2 == self.height - 1 and 0 < col2 < self.width - 1:
            return self.start_local, -100.0
        local2 = self.pos_to_local(row2, col2)
        if local2 == self.goal_local:
            return local2, 0.0
        return local2, -1.0

    def _build_dynamics(self) -> None:
        for state in range(self.states):
            local1, local2 = self.decode(state)
            for action1 in range(self.actions):
                next1, reward1 = self._move(local1, action1)
                for action2 in range(self.actions):
                    next2, reward2 = self._move(local2, action2)
                    if next1 == self.goal_local and next2 == self.goal_local:
                        next1_final = self.start_local
                        next2_final = self.start_local
                        reward1 = 0.0
                        reward2 = 0.0
                    else:
                        next1_final = next1
                        next2_final = next2
                    self.next_state[state, action1, action2] = self.encode(
                        next1_final, next2_final
                    )
                    self.reward[state, action1, action2] = (reward1, reward2)

    def _preferred_action(self, local_state: int) -> int | None:
        if local_state == self.goal_local:
            return None
        row, col = self.local_to_pos(local_state)
        if row == self.height - 1 and col == 0:
            return 0  # leave the start by moving up, not into the cliff
        if col < self.width - 1:
            return 3  # move right above the cliff
        if row < self.height - 1:
            return 1  # descend to the goal in the last column
        return None

    def _build_target_policy(self, epsilon: float) -> np.ndarray:
        policy = np.empty((self.agents, self.states, self.actions), dtype=float)
        for agent in range(self.agents):
            for state in range(self.states):
                local = self.decode(state)[agent]
                preferred = self._preferred_action(local)
                if preferred is None:
                    probs = np.full(self.actions, 1.0 / self.actions)
                else:
                    probs = np.full(self.actions, epsilon / self.actions)
                    probs[preferred] += 1.0 - epsilon
                policy[agent, state] = probs
        return policy

    def behavior_transition_matrix(self) -> np.ndarray:
        matrix = np.zeros((self.states, self.states), dtype=float)
        joint_prob = 1.0 / (self.actions**self.agents)
        for state in range(self.states):
            for action1 in range(self.actions):
                for action2 in range(self.actions):
                    matrix[state, self.next_state[state, action1, action2]] += joint_prob
        return matrix

    def sample_trajectory(
        self, rng: np.random.Generator, length: int, burn_in: int
    ) -> dict[str, np.ndarray]:
        state = self.start_state
        for _ in range(burn_in):
            action1 = int(rng.integers(self.actions))
            action2 = int(rng.integers(self.actions))
            state = int(self.next_state[state, action1, action2])

        states = np.empty(length + 1, dtype=np.int16)
        actions = np.empty((length, self.agents), dtype=np.int8)
        rewards = np.empty((length, self.agents), dtype=float)
        local_rho = np.empty((length, self.agents), dtype=float)
        states[0] = state
        behavior_prob = 1.0 / self.actions

        for step in range(length):
            action1 = int(rng.integers(self.actions))
            action2 = int(rng.integers(self.actions))
            actions[step] = (action1, action2)
            rewards[step] = self.reward[state, action1, action2]
            local_rho[step, 0] = self.target[0, state, action1] / behavior_prob
            local_rho[step, 1] = self.target[1, state, action2] / behavior_prob
            state = int(self.next_state[state, action1, action2])
            states[step + 1] = state

        return {
            "states": states,
            "actions": actions,
            "rewards": rewards,
            "local_rho": local_rho,
        }


def stationary_distribution(transition: np.ndarray) -> np.ndarray:
    size = transition.shape[0]
    system = transition.T - np.eye(size)
    rhs = np.zeros(size)
    system[-1] = 1.0
    rhs[-1] = 1.0
    stationary = np.linalg.solve(system, rhs)
    stationary = np.maximum(stationary, 0.0)
    stationary /= stationary.sum()
    return stationary


def fourier_features(env: TwoAgentCliff, stationary: np.ndarray, dim: int) -> np.ndarray:
    if not 1 <= dim <= 20:
        raise ValueError("The cliff feature dimension K must be between 1 and 20")
    # The basis is closed under swapping the two agents.  This prevents the
    # function approximator, rather than the learning rule, from favoring one
    # of two otherwise symmetric agents.
    local_frequencies = (
        (1, 0),
        (0, 1),
        (1, 1),
        (2, 0),
        (0, 2),
        (2, 1),
        (1, 2),
        (2, 2),
    )
    coefficients: list[tuple[int, int, int, int]] = [(0, 0, 0, 0)]
    for row_frequency, col_frequency in local_frequencies:
        coefficients.append((row_frequency, col_frequency, 0, 0))
        coefficients.append((0, 0, row_frequency, col_frequency))
    for row_frequency, col_frequency in ((1, 0), (0, 1), (1, 1)):
        coefficients.append(
            (row_frequency, col_frequency, row_frequency, col_frequency)
        )

    # K 可以在 1 到 20 之间调节；这里只保留前 K 个 Fourier 特征。
    coefficients = coefficients[:dim]
    features = np.empty((env.states, dim), dtype=float)
    for state in range(env.states):
        local1, local2 = env.decode(state)
        row1, col1 = env.local_to_pos(local1)
        row2, col2 = env.local_to_pos(local2)
        coordinates = np.array(
            [
                row1 / (env.height - 1),
                col1 / (env.width - 1),
                row2 / (env.height - 1),
                col2 / (env.width - 1),
            ]
        )
        features[state] = [
            math.cos(math.pi * np.dot(coeff, coordinates))
            for coeff in coefficients
        ]

    rms = np.sqrt(np.sum(stationary[:, None] * features**2, axis=0))
    features /= np.maximum(rms, 1e-12)
    features /= math.sqrt(dim)
    return features


def evaluation_samples(total: int, points: int, batch_size: int) -> np.ndarray:
    positive = np.geomspace(batch_size, total, max(points - 1, 2))
    rounded = np.rint(positive / batch_size).astype(int) * batch_size
    rounded = np.clip(rounded, batch_size, total)
    return np.unique(np.concatenate(([0], rounded, [total]))).astype(int)


class TwoStateOffPolicy:
    """Small diagnostic where the expected ordinary off-policy TD drift is unstable.

    Agent 0 rarely takes the action that reaches the high-feature state under
    behavior, while the target takes it often. Agent 1 has a private reward
    dependent on its own action. Both local actions enter the joint ratio.
    """

    states = 2
    actions = 2
    agents = 2
    start_state = 0

    def __init__(self) -> None:
        self.next_state = np.zeros((2, 2, 2), dtype=np.int16)
        self.reward = np.zeros((2, 2, 2, 2), dtype=float)
        for s in range(2):
            for a0 in range(2):
                for a1 in range(2):
                    self.next_state[s, a0, a1] = a0 if s == 0 else 0
                    if s == 0:
                        self.reward[s, a0, a1] = (float(a0), float(a1))
        self.behavior = np.empty((2, 2, 2), dtype=float)
        self.target = np.empty((2, 2, 2), dtype=float)
        self.behavior[0, :, :] = (0.95, 0.05)
        self.behavior[1, :, :] = (0.50, 0.50)
        self.target[0, :, :] = (0.05, 0.95)
        self.target[1, :, :] = (0.05, 0.95)

    def behavior_transition_matrix(self) -> np.ndarray:
        matrix = np.zeros((2, 2))
        for s in range(2):
            for a0 in range(2):
                for a1 in range(2):
                    weight = self.behavior[0, s, a0] * self.behavior[1, s, a1]
                    matrix[s, self.next_state[s, a0, a1]] += weight
        return matrix

    def sample_trajectory(
        self, rng: np.random.Generator, length: int, burn_in: int
    ) -> dict[str, np.ndarray]:
        state = self.start_state
        for _ in range(burn_in):
            a0 = int(rng.choice(2, p=self.behavior[0, state]))
            a1 = int(rng.choice(2, p=self.behavior[1, state]))
            state = int(self.next_state[state, a0, a1])
        states = np.empty(length + 1, dtype=np.int16)
        actions = np.empty((length, 2), dtype=np.int8)
        rewards = np.empty((length, 2), dtype=float)
        local_ratio = np.empty((length, 2), dtype=float)
        states[0] = state
        for t in range(length):
            a0 = int(rng.choice(2, p=self.behavior[0, state]))
            a1 = int(rng.choice(2, p=self.behavior[1, state]))
            actions[t] = (a0, a1)
            rewards[t] = self.reward[state, a0, a1]
            local_ratio[t, 0] = self.target[0, state, a0] / self.behavior[0, state, a0]
            local_ratio[t, 1] = self.target[1, state, a1] / self.behavior[1, state, a1]
            state = int(self.next_state[state, a0, a1])
            states[t + 1] = state
        return {
            "states": states,
            "actions": actions,
            "rewards": rewards,
            "local_rho": local_ratio,
        }

## 3. Importance ratio、预处理矩阵和理论目标

这一部分完成：ratio 的 log-consensus、预采样矩阵 $M^n$ 的估计、有限交流轮数下的总体矩阵，以及正确联合策略评估解 $\theta^*$ 的计算。

In [ ]:
def graph_matrix(diagonal: float) -> np.ndarray:
    if not 0.5 < diagonal < 1.0:
        raise ValueError("For this two-node experiment, 0.5 < diagonal < 1 is required")
    return np.array([[diagonal, 1.0 - diagonal], [1.0 - diagonal, diagonal]])


def communicated_ratios(local_ratio: np.ndarray, mixing: np.ndarray, rounds: int) -> np.ndarray:
    """Chen's log-consensus: exp(N W**rounds log(local_ratio))."""
    if rounds < 0:
        raise ValueError("rounds must be nonnegative")
    if local_ratio.shape[-1] != mixing.shape[0] or np.any(local_ratio <= 0):
        raise ValueError("local ratios must be positive and match the graph")
    mixed_log = np.log(local_ratio) @ np.linalg.matrix_power(mixing, rounds).T
    return np.exp(mixing.shape[0] * mixed_log)


def estimated_inverse_matrices(
    trajectory: dict[str, np.ndarray],
    features: np.ndarray,
    gamma: float,
    mixing: np.ndarray,
    rounds: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Estimate each M_n from an independent pre-training Markov trajectory."""
    phi = features[trajectory["states"][:-1]]
    difference = phi - gamma * features[trajectory["states"][1:]]
    ratios = communicated_ratios(trajectory["local_rho"], mixing, rounds)
    estimates = np.stack(
        [(phi * ratios[:, n, None]).T @ difference / len(phi) for n in range(mixing.shape[0])]
    )
    conditions = np.linalg.cond(estimates)
    if np.any(~np.isfinite(conditions)) or np.max(conditions) > 1e10:
        raise RuntimeError(f"Pilot A matrices are ill-conditioned: {conditions}")
    return np.linalg.inv(estimates), conditions


def finite_round_population(
    env: TwoAgentCliff | TwoStateOffPolicy,
    features: np.ndarray,
    stationary: np.ndarray,
    gamma: float,
    mixing: np.ndarray,
    rounds: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute the *actual* population A_n,b_n induced by finite log consensus."""
    agents = env.agents
    dim = features.shape[1]
    a = np.zeros((agents, dim, dim))
    b = np.zeros((agents, dim))
    for state in range(env.states):
        phi = features[state]
        for action0 in range(env.actions):
            for action1 in range(env.actions):
                next_state = int(env.next_state[state, action0, action1])
                difference = phi - gamma * features[next_state]
                behavior_weight = (
                    env.behavior[0, state, action0] * env.behavior[1, state, action1]
                )
                local = np.array([
                    env.target[0, state, action0] / env.behavior[0, state, action0],
                    env.target[1, state, action1] / env.behavior[1, state, action1],
                ])
                ratio = communicated_ratios(local[None, :], mixing, rounds)[0]
                weight = stationary[state] * behavior_weight * ratio
                outer = np.outer(phi, difference)
                rewards = env.reward[state, action0, action1]
                a += weight[:, None, None] * outer
                b += weight[:, None] * rewards[:, None] * phi
    return a, b


def joint_population(
    env: TwoAgentCliff | TwoStateOffPolicy,
    features: np.ndarray,
    stationary: np.ndarray,
    gamma: float,
) -> dict[str, np.ndarray]:
    """Exact joint-ratio projected TD equation, with average private reward."""
    dim = features.shape[1]
    a = np.zeros((dim, dim))
    b = np.zeros((env.agents, dim))
    for state in range(env.states):
        phi = features[state]
        for action0 in range(env.actions):
            for action1 in range(env.actions):
                next_state = int(env.next_state[state, action0, action1])
                target_weight = (
                    env.target[0, state, action0] * env.target[1, state, action1]
                )
                weight = stationary[state] * target_weight
                a += weight * np.outer(phi, phi - gamma * features[next_state])
                b += weight * env.reward[state, action0, action1, :, None] * phi
    return {"A_joint": a, "b_joint_agents": b, "theta_joint": np.linalg.solve(a, b.mean(axis=0))}


def population_equilibrium(a: np.ndarray, b: np.ndarray, inverses: np.ndarray) -> np.ndarray:
    """Consensus fixed point of mean M_n(A_n theta - b_n), K factor cancels."""
    left = np.mean(inverses @ a, axis=0)
    right = np.mean((inverses @ b[..., None])[..., 0], axis=0)
    return np.linalg.solve(left, right)


## 4. 三种算法、步长和误差定义

- DeSPTD：预采样得到每个节点的预处理矩阵，在线时随机抽取一列更新。
- Chen TDC：每个在线样本更新一次主变量和辅助变量。
- Decentralized TD(0)：每个在线样本更新一次，作为普通 TD 基线。
- 衰减步长严格使用 (alpha_t=alpha_0/(t+1))，没有平移项。

In [ ]:
@dataclass
class Config:
    environment: str = "cliff"
    runs: int = 10
    samples: int = 10_000
    pilot_samples: int = 999
    pilot_ratio_rounds: int = 7
    online_ratio_rounds: tuple[int, ...] = (7,)
    burn_in: int = 1_000
    plot_every: int = 20
    seed: int = 1_000
    gamma: float = .95
    feature_dim: int = 8
    target_epsilon: float = .80
    mixing_diagonal: float = .70
    sptd_alpha_constant: float = .005
    sptd_alpha_decay0: float = 8.5
    tdc_alpha_constant: float = .01
    tdc_alpha_decay0: float = 9.0
    tdc_beta_constant: float = .001
    tdc_beta_decay0: float = 2.0
    require_pilot_bound: bool = False


def checkpoints_for(total: int, every: int) -> np.ndarray:
    """每隔 every 步记录一次，并确保包含第 0 步和最后一步。"""
    if total < 1 or every < 1:
        raise ValueError("total 和 every 必须为正整数")
    points = np.arange(0, total + 1, every, dtype=int)
    return np.unique(np.append(points, total))


def step_size(constant: float, decay0: float, online_step: int, schedule: str) -> float:
    """常数步长，或严格的 alpha_0/(t+1)。"""
    if schedule == "constant":
        return constant
    if schedule == "one_over_t":
        return decay0 / (online_step + 1.0)
    raise ValueError(f"unknown schedule {schedule}")


def parameter_errors(theta: np.ndarray, reference: np.ndarray) -> np.ndarray:
    """依次返回平均参数误差、共识误差、最差节点误差。"""
    average = theta.mean(axis=0)
    convergence = np.linalg.norm(average - reference)
    consensus = np.linalg.norm(theta - average[None, :], ord="fro") / np.sqrt(len(theta))
    worst_agent = np.max(np.linalg.norm(theta - reference[None, :], axis=1))
    return np.array([convergence, consensus, worst_agent], dtype=float)


def run_sptd(
    trajectory: dict[str, np.ndarray], ratios: np.ndarray, features: np.ndarray,
    mixing: np.ndarray, inverses: np.ndarray, reference: np.ndarray,
    radius: float, cfg: Config, checkpoints: np.ndarray, schedule: str, seed: int,
) -> tuple[np.ndarray, int]:
    """逐样本运行 DeSPTD；每个节点每步随机抽取一个坐标。"""
    agents, dim = inverses.shape[:2]
    rng = np.random.default_rng(seed)
    theta = np.zeros((agents, dim))
    curves = np.empty((len(checkpoints), len(METRICS)))
    curves[0] = parameter_errors(theta, reference)
    point, projection_hits = 1, 0
    for t in range(cfg.samples):
        phi = features[int(trajectory["states"][t])]
        difference = phi - cfg.gamma * features[int(trajectory["states"][t + 1])]
        old = theta
        new = mixing @ old
        alpha = step_size(
            cfg.sptd_alpha_constant, cfg.sptd_alpha_decay0, t, schedule
        )
        for n in range(agents):
            column = int(rng.integers(dim))
            scalar = ratios[t, n] * phi[column] * (
                float(difference @ old[n]) - float(trajectory["rewards"][t, n])
            )
            new[n] -= alpha * inverses[n, :, column] * scalar
            norm = float(np.linalg.norm(new[n]))
            if norm > radius:
                new[n] *= radius / norm
                projection_hits += 1
        theta = new
        if point < len(checkpoints) and t + 1 == checkpoints[point]:
            curves[point] = parameter_errors(theta, reference)
            point += 1
    if point != len(checkpoints):
        raise RuntimeError("DeSPTD missed an evaluation checkpoint")
    return curves, projection_hits


def run_tdc_td0(
    trajectory: dict[str, np.ndarray], ratios: np.ndarray, features: np.ndarray,
    mixing: np.ndarray, reference: np.ndarray, cfg: Config,
    checkpoints: np.ndarray, schedule: str,
) -> tuple[np.ndarray, np.ndarray, bool]:
    """逐样本运行 Chen TDC 和普通 decentralized TD(0)。"""
    agents, dim = mixing.shape[0], features.shape[1]
    theta_tdc = np.zeros((agents, dim))
    auxiliary = np.zeros((agents, dim))
    theta_td0 = np.zeros((agents, dim))
    tdc = np.empty((len(checkpoints), len(METRICS)))
    td0 = np.empty_like(tdc)
    tdc[0] = parameter_errors(theta_tdc, reference)
    td0[0] = parameter_errors(theta_td0, reference)
    point, diverged = 1, False
    for t in range(cfg.samples):
        phi = features[int(trajectory["states"][t])]
        phi_next = features[int(trajectory["states"][t + 1])]
        alpha = step_size(
            cfg.tdc_alpha_constant, cfg.tdc_alpha_decay0, t, schedule
        )
        beta = step_size(
            cfg.tdc_beta_constant, cfg.tdc_beta_decay0, t, schedule
        )
        old_tdc, old_auxiliary, old_td0 = theta_tdc, auxiliary, theta_td0
        new_tdc = mixing @ old_tdc
        new_auxiliary = mixing @ old_auxiliary
        new_td0 = mixing @ old_td0 if not diverged else old_td0
        for n in range(agents):
            rho = float(ratios[t, n])
            reward = float(trajectory["rewards"][t, n])
            delta = reward + cfg.gamma * float(phi_next @ old_tdc[n]) - float(phi @ old_tdc[n])
            aux_value = float(phi @ old_auxiliary[n])
            new_tdc[n] += alpha * (
                rho * delta * phi - cfg.gamma * rho * aux_value * phi_next
            )
            new_auxiliary[n] += beta * (rho * delta * phi - aux_value * phi)
            if not diverged:
                delta0 = reward + cfg.gamma * float(phi_next @ old_td0[n]) - float(phi @ old_td0[n])
                new_td0[n] += alpha * rho * delta0 * phi
        theta_tdc, auxiliary = new_tdc, new_auxiliary
        if np.any(~np.isfinite(theta_tdc)) or np.linalg.norm(theta_tdc) > 1e15:
            theta_tdc[:] = 1e15
        if not diverged:
            if np.any(~np.isfinite(new_td0)) or np.linalg.norm(new_td0) > 1e15:
                diverged = True
            else:
                theta_td0 = new_td0
        if point < len(checkpoints) and t + 1 == checkpoints[point]:
            tdc[point] = parameter_errors(theta_tdc, reference)
            td0[point] = (
                np.full(len(METRICS), 1e15)
                if diverged else parameter_errors(theta_td0, reference)
            )
            point += 1
    if point != len(checkpoints):
        raise RuntimeError("TDC/TD0 missed an evaluation checkpoint")
    return tdc, td0, diverged

## 5. 保存数据和绘图

每一种误差、每一种步长都单独保存为一张图片，不再拼成大图。  
实线是 10 次独立运行的均值，浅色区域是 10%–90% 分位区间。

In [ ]:

# Publication export: preserve the numerical curves and use embedded vector fonts.
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "stix",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7})
def save_data(output: Path, checkpoints: np.ndarray, raw: np.ndarray, cfg: Config) -> None:
    """保存完整轨迹和便于查看的 CSV 汇总。"""
    np.savez_compressed(
        output / "raw_trajectories.npz", schedules=np.array(SCHEDULES),
        ratio_rounds=np.array(cfg.online_ratio_rounds), algorithms=np.array(ALGORITHMS),
        metrics=np.array(METRICS), samples=checkpoints, curves=raw,
    )
    with (output / "summary_trajectories.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(("schedule", "T_prime", "algorithm", "Step", "metric", "mean", "median", "q10", "q90"))
        for s, schedule in enumerate(SCHEDULES):
            for ell, rounds in enumerate(cfg.online_ratio_rounds):
                for a, algorithm in enumerate(ALGORITHMS):
                    for p, t in enumerate(checkpoints):
                        for m, metric in enumerate(METRICS):
                            values = raw[s, ell, a, :, p, m]
                            writer.writerow((
                                schedule, rounds, algorithm, int(t), metric,
                                float(values.mean()), float(np.median(values)),
                                float(np.quantile(values, .10)), float(np.quantile(values, .90)),
                            ))


# 每张图都有一条完全独立的纵轴设置；修改某一条不会影响任何其他图片。
# 格式：(环境/网络, 步长类型, 误差类型): (纵轴下限, 纵轴上限, 主刻度间隔)
Y_AXIS_SETTINGS = {
    # Cliff：常数步长，convergence error = 平均参数与 theta* 的距离。
    ("cliff", "constant", "mean_parameter_error"): (0.0, 1.2, 0.2),
    # Cliff：常数步长，consensus error = 各节点之间的不一致程度。
    ("cliff", "constant", "consensus_error"): (0.0, 0.14, 0.02),
    # Cliff：常数步长，max error = 最差节点与 theta* 的距离。
    ("cliff", "constant", "max_agent_error"): (0.0, 1.2, 0.2),
    # Cliff：1/t 步长，convergence error。
    ("cliff", "one_over_t", "mean_parameter_error"): (0.0, 2.5, 0.5),
    # Cliff：1/t 步长，consensus error。
    ("cliff", "one_over_t", "consensus_error"): (0.0, 2.5, 0.5),
    # Cliff：1/t 步长，max error。
    ("cliff", "one_over_t", "max_agent_error"): (0.0, 3.0, 0.5),

    # 两状态环境：常数步长，convergence error。
    ("unstable_two_state", "constant", "mean_parameter_error"): (0.0, 1.2, 0.2),
    # 两状态环境：常数步长，consensus error。
    ("unstable_two_state", "constant", "consensus_error"): (0.0, 0.0025, 0.0005),
    # 两状态环境：常数步长，max error。
    ("unstable_two_state", "constant", "max_agent_error"): (0.0, 1.2, 0.2),
    # 两状态环境：1/t 步长，convergence error。
    ("unstable_two_state", "one_over_t", "mean_parameter_error"): (0.0, 2.5, 0.5),
    # 两状态环境：1/t 步长，consensus error。
    ("unstable_two_state", "one_over_t", "consensus_error"): (0.0, 0.014, 0.002),
    # 两状态环境：1/t 步长，max error。
    ("unstable_two_state", "one_over_t", "max_agent_error"): (0.0, 2.5, 0.5),

    # Synthetic MDP 全连接网络：常数步长，convergence error。
    ("synthetic_fully_connected", "constant", "mean_parameter_error"): (0.0, 1.0, 0.2),
    # Synthetic MDP 全连接网络：常数步长，consensus error。
    ("synthetic_fully_connected", "constant", "consensus_error"): (0.0, 0.30, 0.05),
    # Synthetic MDP 全连接网络：常数步长，max error。
    ("synthetic_fully_connected", "constant", "max_agent_error"): (0.0, 1.2, 0.2),
    # Synthetic MDP 全连接网络：1/t 步长，convergence error。
    ("synthetic_fully_connected", "one_over_t", "mean_parameter_error"): (0.0, 1.5, 0.3),
    # Synthetic MDP 全连接网络：1/t 步长，consensus error。
    ("synthetic_fully_connected", "one_over_t", "consensus_error"): (0.0, 2.5, 0.5),
    # Synthetic MDP 全连接网络：1/t 步长，max error。
    ("synthetic_fully_connected", "one_over_t", "max_agent_error"): (0.0, 4.0, 0.5),

    # Synthetic MDP 环形网络：常数步长，convergence error。
    ("synthetic_ring", "constant", "mean_parameter_error"): (0.0, 1.0, 0.2),
    # Synthetic MDP 环形网络：常数步长，consensus error。
    ("synthetic_ring", "constant", "consensus_error"): (0.0, 0.40, 0.05),
    # Synthetic MDP 环形网络：常数步长，max error。
    ("synthetic_ring", "constant", "max_agent_error"): (0.0, 1.2, 0.2),
    # Synthetic MDP 环形网络：1/t 步长，convergence error。
    ("synthetic_ring", "one_over_t", "mean_parameter_error"): (0.0, 1.2, 0.2),
    # Synthetic MDP 环形网络：1/t 步长，consensus error。
    ("synthetic_ring", "one_over_t", "consensus_error"): (0.0, 2.0, 0.25),
    # Synthetic MDP 环形网络：1/t 步长，max error。
    ("synthetic_ring", "one_over_t", "max_agent_error"): (0.0, 4.0, 0.5),
}


# 六段绘图任务逐一列出。每一项都明确对应“哪种步长 + 哪个误差量”。
FIGURE_SPECS = (
    # 图 1：常数步长下的 convergence error。
    ("constant", 0, "Constant step size", "constant", "convergence_error"),
    # 图 2：常数步长下的 consensus error。
    ("constant", 1, "Constant step size", "constant", "consensus_error"),
    # 图 3：常数步长下的最差节点 max error。
    ("constant", 2, "Constant step size", "constant", "total_error"),
    # 图 4：alpha_t=alpha_0/(t+1) 下的 convergence error。
    ("one_over_t", 0, r"$\alpha_t=\alpha_0/(t+1)$", "decay", "convergence_error"),
    # 图 5：alpha_t=alpha_0/(t+1) 下的 consensus error。
    ("one_over_t", 1, r"$\alpha_t=\alpha_0/(t+1)$", "decay", "consensus_error"),
    # 图 6：alpha_t=alpha_0/(t+1) 下的最差节点 max error。
    ("one_over_t", 2, r"$\alpha_t=\alpha_0/(t+1)$", "decay", "total_error"),
)


def draw_individual_plots(
    output: Path, checkpoints: np.ndarray, raw: np.ndarray,
    reference: np.ndarray, cfg: Config, plot_profile: str | None = None,
) -> list[Path]:
    """分别画出 2 种步长 × 3 种误差，共 6 张独立图片。"""
    scale = max(float(np.linalg.norm(reference)), 1e-12)
    profile = cfg.environment if plot_profile is None else plot_profile
    ylabels = (
        r"$\|\bar{\theta}_t-\theta^\ast\|_2/\|\theta^\ast\|_2$",
        r"$N^{-1/2}\|\Theta_t-\mathbf{1}\bar{\theta}_t^\top\|_F/\|\theta^\ast\|_2$",
        r"$\max_n\|\theta_t^n-\theta^\ast\|_2/\|\theta^\ast\|_2$",
    )
    x_step = max(cfg.samples // 5, 1)
    x_ticks = np.arange(0, cfg.samples + 1, x_step)
    saved: list[Path] = []

    for schedule_key, metric, title, file_prefix, file_metric in FIGURE_SPECS:
        schedule_index = SCHEDULES.index(schedule_key)
        metric_key = METRICS[metric]
        axis_key = (profile, schedule_key, metric_key)
        if axis_key not in Y_AXIS_SETTINGS:
            raise KeyError(f"缺少这张图的独立纵轴设置：{axis_key}")
        y_min, y_max, y_step = Y_AXIS_SETTINGS[axis_key]
        y_ticks = np.arange(y_min, y_max + 0.5 * y_step, y_step)
        values_for_axis = raw[schedule_index, 0, :, :, :, metric] / scale

        fig, axis = plt.subplots(figsize=(3.2, 2.5), constrained_layout=True)
        for algorithm, (label, color, style) in enumerate(
            zip(ALGORITHMS, COLORS, STYLES)
        ):
            values = values_for_axis[algorithm]
            mean = values.mean(axis=0)
            #q10, q90 = np.quantile(values, [.10, .90], axis=0)
            axis.plot(
                checkpoints, mean, color=color, linestyle=style,
                linewidth=1.1, label=label, zorder=3,
            )
            #axis.fill_between(
            #    checkpoints, q10, q90,
            #    color=color, alpha=.11, linewidth=0,
            #)

        # 这里只读取当前图片自己的设置，不与其他图片共用纵轴范围。
        axis.set_ylim(y_min, y_max)
        axis.set_yticks(y_ticks)
        # 左侧保留约 2% 空白，使横轴刻度 0 不紧贴纵轴。
        axis.set_xlim(-0.02 * cfg.samples, cfg.samples)
        axis.set_xticks(x_ticks)
        axis.grid(True, which="major", color="#c7ccd1", alpha=.62, linewidth=.8)
        axis.spines[["top", "right"]].set_visible(False)
        axis.set_title(title, fontsize=8, pad=10)
        axis.set_xlabel("Step", fontsize=8)
        axis.set_ylabel(ylabels[metric], fontsize=8)
        axis.tick_params(labelsize=7)
        axis.legend(loc="upper right", frameon=False, fontsize=7)
        path = output / f"{file_prefix}_{file_metric}.png"
        fig.savefig(path, dpi=800, bbox_inches="tight")
        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
        plt.close(fig)
        saved.append(path)
    return saved


## 6. 完整实验流程

每次独立运行先生成 (T_0) 个预采样，再生成 10,000 个正式在线样本。预采样不计入横轴。

In [ ]:
def run_experiment(cfg: Config, output: Path) -> dict[str, object]:
    """运行一个环境的完整比较，并保存图、数据和诊断信息。"""
    cliff = cfg.environment == "cliff"
    ratio_rounds = tuple(sorted(set(cfg.online_ratio_rounds)))
    cfg.online_ratio_rounds = ratio_rounds
    if cfg.runs < 1 or cfg.samples < 1 or cfg.pilot_samples < 1:
        raise ValueError("重复次数、正式步数和预采样数必须为正数")
    if len(ratio_rounds) != 1:
        raise ValueError("这张精简图一次只画一个正式交流轮数 T'")
    if cfg.pilot_ratio_rounds > ratio_rounds[0]:
        raise ValueError("要求 T_0' 不超过正式运行的 T'")
    if not np.isclose(cfg.sptd_alpha_constant, cfg.tdc_alpha_constant):
        raise ValueError("三种方法的常数 alpha 必须完全相同")
    if not np.isclose(cfg.sptd_alpha_decay0, cfg.tdc_alpha_decay0):
        raise ValueError("三种方法的衰减 alpha_0 必须完全相同")
    if cfg.sptd_alpha_decay0 <= cfg.feature_dim:
        raise ValueError("DeSPTD 的衰减步长必须满足 alpha_0 > K")
    if cfg.tdc_alpha_decay0 <= cfg.feature_dim:
        raise ValueError("TDC/TD(0) 的衰减步长必须满足 alpha_0 > K")
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)

    mixing = graph_matrix(cfg.mixing_diagonal)
    env = TwoAgentCliff(cfg.target_epsilon) if cliff else TwoStateOffPolicy()
    stationary = stationary_distribution(env.behavior_transition_matrix())
    features = (
        fourier_features(env, stationary, cfg.feature_dim)
        if cliff else np.array([[.2], [1.]])
    )
    population = joint_population(env, features, stationary, cfg.gamma)
    reference, exact_A = population["theta_joint"], population["A_joint"]
    scale = max(float(np.linalg.norm(reference)), 1e-12)
    radius = max(3.0 * scale, 10.0)
    checkpoints = checkpoints_for(cfg.samples, cfg.plot_every)

    raw = np.empty((len(SCHEDULES), 1, len(ALGORITHMS), cfg.runs,
                    len(checkpoints), len(METRICS)))
    inverse_residuals = np.empty((cfg.runs, env.agents))
    projection_hits = np.empty((cfg.runs, len(SCHEDULES)), dtype=int)
    td0_diverged = np.empty((cfg.runs, len(SCHEDULES)), dtype=bool)
    ratio_relative_error = np.empty(cfg.runs)

    rounds = ratio_rounds[0]
    for run in range(cfg.runs):
        pilot = env.sample_trajectory(
            np.random.default_rng(cfg.seed + run),
            cfg.pilot_samples, cfg.burn_in,
        )
        online = env.sample_trajectory(
            np.random.default_rng(cfg.seed + 1_000 + run),
            cfg.samples, cfg.burn_in,
        )
        inverses, _ = estimated_inverse_matrices(
            pilot, features, cfg.gamma, mixing, cfg.pilot_ratio_rounds
        )
        inverse_residuals[run] = [
            np.linalg.norm(inverse @ exact_A - np.eye(cfg.feature_dim), 2)
            for inverse in inverses
        ]
        if cfg.require_pilot_bound and inverse_residuals[run].max() > .5:
            raise RuntimeError("预处理矩阵误差超过 0.5，请增加 T_0")

        exact_ratio = np.prod(online["local_rho"], axis=1)
        ratios = communicated_ratios(online["local_rho"], mixing, rounds)
        ratio_relative_error[run] = float(np.max(np.abs(
            ratios / exact_ratio[:, None] - 1.0
        )))

        for s, schedule in enumerate(SCHEDULES):
            raw[s, 0, 0, run], projection_hits[run, s] = run_sptd(
                online, ratios, features, mixing, inverses, reference, radius,
                cfg, checkpoints, schedule,
                cfg.seed + 2_000 + 10 * run + s,
            )
            raw[s, 0, 1, run], raw[s, 0, 2, run], td0_diverged[run, s] = run_tdc_td0(
                online, ratios, features, mixing, reference, cfg,
                checkpoints, schedule,
            )
        print(
            f"{cfg.environment}: run {run + 1}/{cfg.runs}; "
            f"max pilot residual={inverse_residuals[run].max():.3f}; "
            f"decay final DeSPTD={raw[1, 0, 0, run, -1, 0] / scale:.3g}",
            flush=True,
        )

    if np.any(~np.isfinite(raw)):
        raise RuntimeError("结果中出现非有限数")
    save_data(output, checkpoints, raw, cfg)
    figure_paths = draw_individual_plots(output, checkpoints, raw, reference, cfg)

    (output / "config.json").write_text(
        json.dumps(asdict(cfg), indent=2, ensure_ascii=False), encoding="utf-8"
    )
    diagnostics = {
        "theta_dimension_K": cfg.feature_dim,
        "theta_norm": scale,
        "max_pilot_inverse_residual": float(inverse_residuals.max()),
        "pilot_inverse_residuals": inverse_residuals.tolist(),
        "max_ratio_relative_error": float(ratio_relative_error.max()),
        "projection_hits": projection_hits.tolist(),
        "td0_diverged": td0_diverged.tolist(),
        "normalized_errors_at_selected_steps": {},
        "note": "T0 samples and their communication are excluded from Step.",
    }
    selected = (0, 1_000, 2_000, 5_000, cfg.samples)
    for s, schedule in enumerate(SCHEDULES):
        diagnostics["normalized_errors_at_selected_steps"][schedule] = {}
        for a, algorithm in enumerate(ALGORITHMS):
            diagnostics["normalized_errors_at_selected_steps"][schedule][algorithm] = {
                str(step): float(raw[s, 0, a, :, np.flatnonzero(checkpoints == step)[0], 0].mean() / scale)
                for step in selected if np.any(checkpoints == step)
            }
    (output / "diagnostics.json").write_text(
        json.dumps(diagnostics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"已保存：{output.resolve()}")
    return {
        "checkpoints": checkpoints,
        "raw": raw,
        "reference": reference,
        "diagnostics": diagnostics,
        "output": output,
        "figure_paths": figure_paths,
    }

## 7. 参数区（本地调参只改下一格）

`TOTAL_STEPS` 控制横轴终点；`CLIFF_T0`/`TWO_STATE_T0`、`T0_RATIO_ROUNDS`、`ONLINE_RATIO_ROUNDS` 分别对应 (T_0,T_0',T')。  
同一环境内三种方法共用相同的 (alpha,alpha_0)。每个环境分别设置常数步长和衰减步长下的 beta/alpha 比例。

In [ ]:
# ==================== 最常修改的参数：从这里调参 ====================
ENVIRONMENTS = ("cliff", "unstable_two_state")
RUNS = 10                 # 独立重复次数；越大曲线越平滑，但运行越慢
TOTAL_STEPS = 10_000      # 正式在线步数，也是横轴 Step 的终点
PLOT_EVERY = 20           # 每隔多少步保存一个绘图点
CLIFF_T0 = 500            # Cliff 中经验上可以稳定运行的较小 T_0（不计入横轴）
TWO_STATE_T0 = 999        # 增大步长后仍保持稳定所需的 T_0（不计入横轴）
T0_RATIO_ROUNDS = 7       # T_0'：沿用此前稳定设置
ONLINE_RATIO_ROUNDS = (7,)  # 正式运行每一步的 communication 次数 T'
SEED = 1_000

# Cliff：三种方法共用 alpha；DeTDC 保持 Chen 实验中的 beta=0.01*alpha
CLIFF_PARAMETERS = dict(
    feature_dim=8,
    target_epsilon=.80,
    alpha_constant=.005,
    alpha_decay0=10.0,  # 满足 alpha_0 > K=8
    beta_constant_ratio=.01,
    beta_decay_ratio=.01,
)

# 两状态诊断环境：这组比例让 DeSPTD、DeTDC 收敛，而 DeTD(0) 发散
TWO_STATE_PARAMETERS = dict(
    feature_dim=1,
    target_epsilon=.80,  # 该环境不使用这个参数，只为保持配置格式一致
    alpha_constant=.002,
    alpha_decay0=3.0,  # 满足 alpha_0 > K=1
    beta_constant_ratio=5.0,  # beta=0.01=5*alpha
    beta_decay_ratio=2.0,     # beta_0=6=2*alpha_0
)


def make_config(environment: str) -> Config:
    if environment not in ENVIRONMENTS:
        raise ValueError(f"未知环境：{environment}")
    chosen = CLIFF_PARAMETERS if environment == "cliff" else TWO_STATE_PARAMETERS
    alpha = chosen["alpha_constant"]
    alpha0 = chosen["alpha_decay0"]
    return Config(
        environment=environment,
        runs=RUNS,
        samples=TOTAL_STEPS,
        pilot_samples=CLIFF_T0 if environment == "cliff" else TWO_STATE_T0,
        pilot_ratio_rounds=T0_RATIO_ROUNDS,
        online_ratio_rounds=ONLINE_RATIO_ROUNDS,
        burn_in=1_000,
        plot_every=PLOT_EVERY,
        seed=SEED,
        gamma=.95,
        feature_dim=chosen["feature_dim"],
        target_epsilon=chosen["target_epsilon"],
        mixing_diagonal=.70,
        sptd_alpha_constant=alpha,
        sptd_alpha_decay0=alpha0,
        tdc_alpha_constant=alpha,
        tdc_alpha_decay0=alpha0,
        tdc_beta_constant=chosen["beta_constant_ratio"] * alpha,
        tdc_beta_decay0=chosen["beta_decay_ratio"] * alpha0,
        require_pilot_bound=False,
    )


configs = {environment: make_config(environment) for environment in ENVIRONMENTS}
output_dirs = {
    "cliff": Path("individual_cliff_result"),
    "unstable_two_state": Path("individual_unstable_two_state_result"),
}

for environment, cfg in configs.items():
    print(f"\n环境：{environment}")
    print(f"  K={cfg.feature_dim}, Step=0...{cfg.samples:,}, runs={cfg.runs}")
    print(f"  T_0={cfg.pilot_samples}, T_0'={cfg.pilot_ratio_rounds}, T'={cfg.online_ratio_rounds[0]}")
    print(f"  三种方法共用：常数 alpha={cfg.sptd_alpha_constant}, 衰减 alpha_0={cfg.sptd_alpha_decay0}")
    print(f"  DeTDC beta：常数 {cfg.tdc_beta_constant}, 衰减 beta_0={cfg.tdc_beta_decay0}")
    print(f"  beta/alpha：常数 {cfg.tdc_beta_constant / cfg.tdc_alpha_constant}, 衰减 {cfg.tdc_beta_decay0 / cfg.tdc_alpha_decay0}")

## 8. 运行两个环境

每个环境会输出 6 张互相独立的图片：2 种步长 × 3 种误差。

In [ ]:
all_results = {}
for environment in ENVIRONMENTS:
    print(f"\n{'=' * 18} 开始：{environment} {'=' * 18}")
    all_results[environment] = run_experiment(
        configs[environment], output_dirs[environment]
    )
print("\n两个环境均已运行完成。")

## 9. 打印最终采用的参数和关键数值

In [ ]:
for environment in ENVIRONMENTS:
    cfg = configs[environment]
    diagnostic = all_results[environment]["diagnostics"]
    print(f"\n[{environment}]")
    print(f"K={cfg.feature_dim}; T_0={cfg.pilot_samples}; T_0'={cfg.pilot_ratio_rounds}; T'={cfg.online_ratio_rounds[0]}")
    print(f"三种方法统一：常数 alpha={cfg.sptd_alpha_constant}; 衰减 alpha_0={cfg.sptd_alpha_decay0}")
    print(f"DeTDC：常数 beta={cfg.tdc_beta_constant}; 衰减 beta_0={cfg.tdc_beta_decay0}")
    print(f"最大预处理残差（仅作诊断，不作为本实验硬约束）：{diagnostic['max_pilot_inverse_residual']:.4f}")
    print(json.dumps(diagnostic["normalized_errors_at_selected_steps"], indent=2, ensure_ascii=False))

## 10. 显示所有独立图片

图片顺序为：常数步长的 convergence、consensus、总误差，然后是衰减步长的三个对应误差。

In [ ]:
from IPython.display import Image, Markdown, display

figure_names = (
    "constant_convergence_error.png",
    "constant_consensus_error.png",
    "constant_total_error.png",
    "decay_convergence_error.png",
    "decay_consensus_error.png",
    "decay_total_error.png",
)
for environment in ENVIRONMENTS:
    display(Markdown(f"## {environment}"))
    for figure_name in figure_names:
        display(Markdown(f"### {figure_name.removesuffix('.png').replace('_', ' ')}"))
        display(Image(filename=str(output_dirs[environment] / figure_name)))

## 11. 打包结果（可选）

In [ ]:
archives = {}
for environment in ENVIRONMENTS:
    directory = output_dirs[environment]
    archives[environment] = shutil.make_archive(
        str(directory), "zip", root_dir=directory
    )
    print(f"{environment}: {Path(archives[environment]).resolve()}")

## 本地调参速查

只需回到 **第 7 节“参数区”**：

- 横轴长度：`TOTAL_STEPS`。
- 预采样：`CLIFF_T0` 和 `TWO_STATE_T0`。
- ratio 交流：`T0_RATIO_ROUNDS` 和 `ONLINE_RATIO_ROUNDS`，前者不得大于后者。
- 三种方法共同的步长：两个环境参数表中的 `alpha_constant`、`alpha_decay0`。
- DeTDC 的辅助步长比例：在两个环境参数表中分别修改 `beta_constant_ratio` 和 `beta_decay_ratio`。
- 曲线平滑程度：`RUNS`。

如果只想运行一个环境，把 `ENVIRONMENTS` 改为 `("cliff",)` 或 `("unstable_two_state",)`。

## 12. Chen 等（2021）第 5.1 节：10-agent 人造环境

这一部分复现论文第 5.1 节的多智能体人造环境：10 个智能体共享 10 个状态，每个智能体有 2 个动作；行为策略为均匀策略，目标策略由均匀策略加标准差 0.05 的高斯扰动后归一化得到；转移概率和各节点奖励从 `[0,1]` 均匀分布生成；5 维状态特征由标准高斯生成并归一化；折扣因子为 0.95。

每个节点先持有自己的 **local importance ratio**，再按照 Chen 文中的 log-consensus 做有限轮通信。预采样阶段使用 $T_0'$ 轮，正式迭代的每个样本使用 $T'$ 轮；它们都大于 1。有限轮通信得到的是全局 importance ratio 的分布式近似，而不是直接读取精确的全局乘积。

实验分别使用 fully-connected network 和 ring network，每个 network 的六张图都有独立纵轴设置。


In [ ]:
class SyntheticMDP:
    """Chen et al. (2021), Section 5.1 synthetic multi-agent MDP."""

    states = 10
    actions = 2
    agents = 10
    start_state = 0

    def __init__(self, seed: int = 20_210_501, target_noise_std: float = .05):
        rng = np.random.default_rng(seed)
        self.seed = seed
        self.target_noise_std = target_noise_std
        self.joint_action_count = self.actions ** self.agents
        indices = np.arange(self.joint_action_count, dtype=np.int32)
        bit_positions = np.arange(self.agents, dtype=np.int32)
        self.joint_action_bits = ((indices[:, None] >> bit_positions) & 1).astype(np.int8)
        self.joint_action_weights = (1 << bit_positions).astype(np.int32)

        transition_raw = rng.uniform(
            0.0, 1.0, size=(self.states, self.joint_action_count, self.states)
        )
        self.transition = transition_raw / transition_raw.sum(axis=2, keepdims=True)
        self.reward = rng.uniform(
            0.0, 1.0,
            size=(self.states, self.joint_action_count, self.states, self.agents),
        )

        self.behavior = np.full(
            (self.agents, self.states, self.actions), 1.0 / self.actions
        )
        perturbed = self.behavior + rng.normal(
            0.0, target_noise_std, size=self.behavior.shape
        )
        # 很小的截断只用于确保每个动作概率严格为正，然后按状态归一化。
        perturbed = np.maximum(perturbed, 1e-6)
        self.target = perturbed / perturbed.sum(axis=2, keepdims=True)

        features = rng.normal(size=(self.states, 5))
        self.features = features / np.linalg.norm(features, axis=1, keepdims=True)

    def behavior_transition_matrix(self) -> np.ndarray:
        # 行为策略独立且均匀，因此每个联合动作的概率均为 2^{-10}。
        return self.transition.mean(axis=1)

    def sample_trajectory(
        self, rng: np.random.Generator, length: int, burn_in: int
    ) -> dict[str, np.ndarray]:
        state = self.start_state
        for _ in range(burn_in):
            actions = rng.integers(self.actions, size=self.agents, dtype=np.int8)
            joint_action = int(actions @ self.joint_action_weights)
            state = int(rng.choice(self.states, p=self.transition[state, joint_action]))

        states = np.empty(length + 1, dtype=np.int16)
        actions = np.empty((length, self.agents), dtype=np.int8)
        rewards = np.empty((length, self.agents), dtype=float)
        local_ratio = np.empty((length, self.agents), dtype=float)
        states[0] = state

        for t in range(length):
            action = rng.integers(self.actions, size=self.agents, dtype=np.int8)
            joint_action = int(action @ self.joint_action_weights)
            next_state = int(
                rng.choice(self.states, p=self.transition[state, joint_action])
            )
            actions[t] = action
            rewards[t] = self.reward[state, joint_action, next_state]
            local_ratio[t] = (
                self.target[np.arange(self.agents), state, action]
                / self.behavior[np.arange(self.agents), state, action]
            )
            states[t + 1] = next_state
            state = next_state

        return {
            "states": states,
            "actions": actions,
            "rewards": rewards,
            "local_rho": local_ratio,
        }


def synthetic_network_matrix(network: str, agents: int = 10) -> np.ndarray:
    """The two doubly-stochastic matrices used in Chen Section 5.1."""
    if agents != 10:
        raise ValueError("Chen Section 5.1 uses exactly 10 agents")
    if network == "fully_connected":
        matrix = np.full((agents, agents), 1.0 / 45.0)
        np.fill_diagonal(matrix, .8)
    elif network == "ring":
        matrix = np.zeros((agents, agents))
        for n in range(agents):
            matrix[n, n] = .8
            matrix[n, (n - 1) % agents] = .1
            matrix[n, (n + 1) % agents] = .1
    else:
        raise ValueError(f"unknown Chen network: {network}")
    if not np.allclose(matrix.sum(axis=0), 1.0) or not np.allclose(
        matrix.sum(axis=1), 1.0
    ):
        raise RuntimeError("Chen communication matrix is not doubly stochastic")
    return matrix


def synthetic_population(
    env: SyntheticMDP, stationary: np.ndarray, gamma: float
) -> dict[str, np.ndarray]:
    """Exact A, local b_n and theta* under the joint target policy."""
    features = env.features
    dim = features.shape[1]
    a = np.zeros((dim, dim))
    b = np.zeros((env.agents, dim))
    agent_index = np.arange(env.agents)

    for state in range(env.states):
        phi = features[state]
        target_action_prob = env.target[
            agent_index[:, None], state, env.joint_action_bits.T
        ].prod(axis=0)
        for joint_action in range(env.joint_action_count):
            weight = stationary[state] * target_action_prob[joint_action]
            transition = env.transition[state, joint_action]
            expected_next_feature = transition @ features
            a += weight * np.outer(phi, phi - gamma * expected_next_feature)
            expected_reward = transition @ env.reward[state, joint_action]
            b += weight * expected_reward[:, None] * phi[None, :]

    return {
        "A_joint": a,
        "b_joint_agents": b,
        "theta_joint": np.linalg.solve(a, b.mean(axis=0)),
    }


def synthetic_inverse_matrices(
    trajectory: dict[str, np.ndarray], features: np.ndarray,
    gamma: float, mixing: np.ndarray, rounds: int,
) -> tuple[np.ndarray, np.ndarray]:
    """用预采样轨迹和 T_0' 轮 ratio communication 估计每个 M_n。"""
    phi = features[trajectory["states"][:-1]]
    difference = phi - gamma * features[trajectory["states"][1:]]
    communicated = communicated_ratios(trajectory["local_rho"], mixing, rounds)
    estimates = np.stack([
        (phi * communicated[:, n, None]).T @ difference / len(phi)
        for n in range(mixing.shape[0])
    ])
    conditions = np.linalg.cond(estimates)
    if np.any(~np.isfinite(conditions)) or np.max(conditions) > 1e10:
        raise RuntimeError(f"Chen communicated-ratio pilot A is ill-conditioned: {conditions}")
    return np.linalg.inv(estimates), conditions


In [ ]:
def prepare_synthetic_runs(
    env: SyntheticMDP, cfg: Config,
) -> list[tuple[dict[str, np.ndarray], dict[str, np.ndarray]]]:
    """预先生成轨迹；两个网络复用完全相同的轨迹，以保证比较公平。"""
    prepared = []
    for run in range(cfg.runs):
        pilot = env.sample_trajectory(
            np.random.default_rng(cfg.seed + run), cfg.pilot_samples, cfg.burn_in
        )
        online = env.sample_trajectory(
            np.random.default_rng(cfg.seed + 1_000 + run), cfg.samples, cfg.burn_in
        )
        prepared.append((pilot, online))
    return prepared


def run_synthetic_network(
    env: SyntheticMDP, network: str, cfg: Config,
    prepared: list[tuple[dict[str, np.ndarray], dict[str, np.ndarray]]],
    exact_a: np.ndarray, reference: np.ndarray, output: Path,
) -> dict[str, object]:
    """在一个 Synthetic MDP 网络上运行三种算法，并使用有限轮 ratio communication。"""
    if cfg.pilot_ratio_rounds <= 1 or cfg.online_ratio_rounds[0] <= 1:
        raise ValueError("Synthetic MDP 要求 T_0' 和 T' 都严格大于 1")
    if cfg.pilot_ratio_rounds > cfg.online_ratio_rounds[0]:
        raise ValueError("要求 Synthetic MDP 的 T_0' 不超过 T'")

    mixing = synthetic_network_matrix(network, env.agents)
    scale = max(float(np.linalg.norm(reference)), 1e-12)
    radius = max(3.0 * scale, 10.0)
    checkpoints = checkpoints_for(cfg.samples, cfg.plot_every)
    raw = np.empty((
        len(SCHEDULES), 1, len(ALGORITHMS), cfg.runs,
        len(checkpoints), len(METRICS),
    ))
    projection_hits = np.empty((cfg.runs, len(SCHEDULES)), dtype=int)
    td0_diverged = np.empty((cfg.runs, len(SCHEDULES)), dtype=bool)
    inverse_residuals = np.empty((cfg.runs, env.agents))
    ratio_relative_error = np.empty(cfg.runs)
    online_rounds = cfg.online_ratio_rounds[0]

    for run, (pilot, online) in enumerate(prepared):
        # 预采样阶段：各节点通过 T_0' 轮通信近似全局 ratio，再估计各自的 M_n。
        inverses, _ = synthetic_inverse_matrices(
            pilot, env.features, cfg.gamma, mixing, cfg.pilot_ratio_rounds
        )
        inverse_residuals[run] = [
            np.linalg.norm(inverse @ exact_a - np.eye(cfg.feature_dim), 2)
            for inverse in inverses
        ]

        # 正式迭代：每个样本的 local ratio 经过 T' 轮 log-consensus。
        ratios = communicated_ratios(online["local_rho"], mixing, online_rounds)
        exact_ratio = np.prod(online["local_rho"], axis=1)
        ratio_relative_error[run] = float(np.max(np.abs(
            ratios / exact_ratio[:, None] - 1.0
        )))

        for s, schedule in enumerate(SCHEDULES):
            raw[s, 0, 0, run], projection_hits[run, s] = run_sptd(
                online, ratios, env.features, mixing, inverses, reference,
                radius, cfg, checkpoints, schedule,
                cfg.seed + 2_000 + 10 * run + s,
            )
            raw[s, 0, 1, run], raw[s, 0, 2, run], td0_diverged[run, s] = (
                run_tdc_td0(
                    online, ratios, env.features, mixing, reference,
                    cfg, checkpoints, schedule,
                )
            )
        print(
            f"Synthetic/{network}: run {run + 1}/{cfg.runs}; "
            f"decay final DeSPTD={raw[1, 0, 0, run, -1, 0] / scale:.3g}",
            flush=True,
        )

    if np.any(~np.isfinite(raw)):
        raise RuntimeError("Synthetic MDP 结果中出现非有限数")
    output.mkdir(parents=True, exist_ok=True)
    save_data(output, checkpoints, raw, cfg)
    figure_paths = draw_individual_plots(
        output, checkpoints, raw, reference, cfg,
        plot_profile=f"synthetic_{network}",
    )
    selected_steps = (0, 1_000, 2_000, 5_000, cfg.samples)
    selected_errors = {}
    for s, schedule in enumerate(SCHEDULES):
        selected_errors[schedule] = {}
        for a, algorithm in enumerate(ALGORITHMS):
            selected_errors[schedule][algorithm] = {
                str(step): float(
                    raw[s, 0, a, :, np.flatnonzero(checkpoints == step)[0], 0].mean()
                    / scale
                )
                for step in selected_steps if np.any(checkpoints == step)
            }

    diagnostics = {
        "source": "Chen et al. (2021), Section 5.1",
        "network": network,
        "agents": env.agents,
        "states": env.states,
        "local_actions": env.actions,
        "joint_actions": env.joint_action_count,
        "feature_dimension_K": cfg.feature_dim,
        "importance_ratio": "local ratios followed by finite-round log-consensus",
        "pilot_ratio_rounds_T0_prime": cfg.pilot_ratio_rounds,
        "online_ratio_rounds_T_prime": online_rounds,
        "reference_solution": "joint target-policy theta*",
        "theta_norm": scale,
        "max_pilot_inverse_residual": float(inverse_residuals.max()),
        "max_ratio_relative_error": float(ratio_relative_error.max()),
        "projection_hits": projection_hits.tolist(),
        "td0_diverged": td0_diverged.tolist(),
        "normalized_errors_at_selected_steps": selected_errors,
    }
    (output / "diagnostics.json").write_text(
        json.dumps(diagnostics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (output / "config.json").write_text(
        json.dumps(
            {
                "algorithm_config": asdict(cfg),
                "environment_seed": env.seed,
                "target_noise_std": env.target_noise_std,
                "network": network,
                "network_matrix": mixing.tolist(),
                "note": "Section 5.1 environment; each local ratio is communicated for finite rounds.",
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print(f"已保存：{output.resolve()}")
    return {
        "checkpoints": checkpoints,
        "raw": raw,
        "reference": reference,
        "diagnostics": diagnostics,
        "output": output,
        "figure_paths": figure_paths,
    }


## 13. Synthetic MDP 参数与正式运行

`CHEN51_*` 是这个环境的独立参数。`CHEN51_T0_RATIO_ROUNDS` 就是预采样通信次数 $T_0'$，`CHEN51_ONLINE_RATIO_ROUNDS` 就是正式迭代每个样本的通信次数 $T'$。两个网络使用同一批 MDP、轨迹和步长，只有通信矩阵不同。


In [ ]:
# ==================== Synthetic MDP 环境的独立参数 ====================
CHEN51_NETWORKS = ("fully_connected", "ring")
CHEN51_RUNS = 10
CHEN51_STEPS = 10_000
CHEN51_PLOT_EVERY = 20
CHEN51_T0 = 800

# T_0'：预采样阶段，每个样本的 local importance ratio 通信轮数。
CHEN51_T0_RATIO_ROUNDS = 5
# T'：正式迭代阶段，每个样本的 local importance ratio 通信轮数。
CHEN51_ONLINE_RATIO_ROUNDS = 5

CHEN51_ENVIRONMENT_SEED = 20_210_501
CHEN51_TRAJECTORY_SEED = 30_000

# 三种方法使用相同的主步长；DeTDC 使用 Chen 实验中的 beta=0.01*alpha。
CHEN51_ALPHA_CONSTANT = 0.02
CHEN51_ALPHA_DECAY0 = 7.0  # K=5，因此满足 alpha_0 > K
CHEN51_BETA_CONSTANT = .01 * CHEN51_ALPHA_CONSTANT
CHEN51_BETA_DECAY0 = .01 * CHEN51_ALPHA_DECAY0

synthetic_cfg = Config(
    environment="synthetic_synthetic",
    runs=CHEN51_RUNS,
    samples=CHEN51_STEPS,
    pilot_samples=CHEN51_T0,
    pilot_ratio_rounds=CHEN51_T0_RATIO_ROUNDS,
    online_ratio_rounds=(CHEN51_ONLINE_RATIO_ROUNDS,),
    burn_in=1_000,
    plot_every=CHEN51_PLOT_EVERY,
    seed=CHEN51_TRAJECTORY_SEED,
    gamma=.95,
    feature_dim=5,
    target_epsilon=.0,
    mixing_diagonal=.8,
    sptd_alpha_constant=CHEN51_ALPHA_CONSTANT,
    sptd_alpha_decay0=CHEN51_ALPHA_DECAY0,
    tdc_alpha_constant=CHEN51_ALPHA_CONSTANT,
    tdc_alpha_decay0=CHEN51_ALPHA_DECAY0,
    tdc_beta_constant=CHEN51_BETA_CONSTANT,
    tdc_beta_decay0=CHEN51_BETA_DECAY0,
    require_pilot_bound=False,
)

synthetic_env = SyntheticMDP(seed=CHEN51_ENVIRONMENT_SEED)
synthetic_stationary = stationary_distribution(
    synthetic_env.behavior_transition_matrix()
)
synthetic_population_values = synthetic_population(
    synthetic_env, synthetic_stationary, synthetic_cfg.gamma
)
synthetic_reference = synthetic_population_values["theta_joint"]
synthetic_prepared = prepare_synthetic_runs(synthetic_env, synthetic_cfg)

synthetic_output_dirs = {
    "fully_connected": Path("synthetic_fully_connected_result"),
    "ring": Path("synthetic_ring_result"),
}
synthetic_results = {}
for network in CHEN51_NETWORKS:
    print(f"\n{'=' * 16} Synthetic MDP：{network} {'=' * 16}")
    synthetic_results[network] = run_synthetic_network(
        synthetic_env,
        network,
        synthetic_cfg,
        synthetic_prepared,
        synthetic_population_values["A_joint"],
        synthetic_reference,
        synthetic_output_dirs[network],
    )

print("\nSynthetic MDP 两种 network 均已运行完成。")
print(
    f"alpha={CHEN51_ALPHA_CONSTANT}, beta={CHEN51_BETA_CONSTANT}; "
    f"alpha_0={CHEN51_ALPHA_DECAY0}, beta_0={CHEN51_BETA_DECAY0}; "
    f"T_0={CHEN51_T0}, T_0'={CHEN51_T0_RATIO_ROUNDS}, "
    f"T'={CHEN51_ONLINE_RATIO_ROUNDS}"
)


## 14. 分开显示并打包两个 network 的图片

In [ ]:
from IPython.display import Image, Markdown, display

for network in CHEN51_NETWORKS:
    display(Markdown(f"## Synthetic MDP - {network}"))
    for figure_name in figure_names:
        display(Markdown(f"### {figure_name.removesuffix('.png').replace('_', ' ')}"))
        display(Image(filename=str(synthetic_output_dirs[network] / figure_name)))

synthetic_archives = {}
for network in CHEN51_NETWORKS:
    directory = synthetic_output_dirs[network]
    synthetic_archives[network] = shutil.make_archive(
        str(directory), "zip", root_dir=directory
    )
    print(f"{network}: {Path(synthetic_archives[network]).resolve()}")

## Synthetic MDP 最常需要修改的参数

- 正式步数：`CHEN51_STEPS`。
- 重复次数：`CHEN51_RUNS`。
- 预采样数量：`CHEN51_T0`。
- 预采样 ratio 通信次数 $T_0'$：`CHEN51_T0_RATIO_ROUNDS`。
- 正式迭代每个样本的 ratio 通信次数 $T'$：`CHEN51_ONLINE_RATIO_ROUNDS`。
- 常数主步长：`CHEN51_ALPHA_CONSTANT`；DeTDC 的辅助步长为 `CHEN51_BETA_CONSTANT`。
- 衰减主步长的系数：`CHEN51_ALPHA_DECAY0`；辅助步长系数为 `CHEN51_BETA_DECAY0`。
- 要运行的网络：`CHEN51_NETWORKS=("fully_connected", "ring")`。
- 每张图的独立纵轴：第 5 节代码中的 `Y_AXIS_SETTINGS`。Chen 全连接使用 `synthetic_fully_connected` 开头的六项，环形网络使用 `synthetic_ring` 开头的六项。


## 投稿用清晰图片：从保存数据快速导出
运行下一个单元即可从归档结果生成矢量 PDF、800 dpi PNG 和组合 TIFF，无需重新运行实验。曲线数值和现有坐标范围保持不变。


In [ ]:
from pathlib import Path
import runpy
for candidate in (Path.cwd(), Path.cwd() / "article-reproduction", Path.cwd().parent):
    if (candidate / "export_figures.py").is_file() and (candidate / "data").is_dir():
        runpy.run_path(str(candidate / "export_figures.py"), run_name="__main__")
        break
else:
    raise FileNotFoundError("Open this notebook from the article-reproduction folder, or its parent.")
